# POSTPROCESS 4K MASTER -- Real-ESRGAN x4 + GFPGAN + RIFE + ProRes 4444

**Setup:** Upload `take_0.mp4` (or any clip) as a Kaggle Dataset -> attach via *Add data* -> set **GPU T4** -> Run All

| Step | What it does | Output res |
|---|---|---|
| hqdn3d denoise | Removes inter-frame grain | source |
| GFPGAN face SR | Reconstructs face detail (weight=0.7) | source |
| Real-ESRGan x4 | Neural texture upscale | source x4 |
| **Temporal blend** (NEW) | Cross-frame avg kills ESRGAN per-frame shimmer | source x4 |
| **RIFE -> 24fps** (NEW) | Optical-flow interpolator (Netflix-grade); kills 16fps choppy feel | source x4 |
| Cinematic grade | Cold grey-blue concrete palette | source x4 |
| Pad to UHD | Exact 3840x2160 16:9 letterbox | 3840x2160 |
| **H.264 review cut** | Small scrubbable mp4 | 3840x2160 (yuv420p) |
| **ProRes 4444 XQ** (NEW) | 10-bit broadcast-grade .mov master -- no gradient banding | 3840x2160 (yuv444p10le) |

**GPU T4:** ~15-20 min total | **CPU only:** ~2-3 hrs

**Fix included:** auto-patches `basicsr/data/degradations.py` for torchvision >= 0.16


In [ ]:
# ==============================================================================
# POSTPROCESS 4K MASTER — take_0.mp4 -> take_0_4k.mp4 + take_0_master.mov
# Steps: temporal denoise -> GFPGAN face SR -> Real-ESRGAN x4 -> temporal
#        blend (kills shimmer) -> RIFE to 24fps -> cinematic grade -> 4K pad
#        -> H.264 review cut + ProRes 4444 XQ 10-bit master
# Kaggle GPU T4 | ~15-20 min total
# SETUP: Upload take_0.mp4 as a Kaggle Dataset and attach via Add data.
# ==============================================================================
import os, sys, subprocess, glob, json, shutil, site
from pathlib import Path

# -------------------- KNOBS --------------------
RIFE_TFPS     = 24       # 0 = skip interpolation
TEMPORAL_BLEND = 0.15    # 0 = off
PRORES_MASTER = True
# ---- Scene-aware grading (set to a dict like {(0, 96): 'cube/cold.cube',
#   (96, 192): 'cube/warm.cube'} }) to apply per-scene .cube LUTs.
# Empty dict -> single global GRADE (unchanged behaviour).
SCENE_LUTS = {}
# ------------------------------------------------

# Auto-locate render-of-take (or any input mp4) from attached dataset
_candidates = glob.glob('/kaggle/input/**/*.mp4', recursive=True)
assert _candidates, (
    'No input mp4 found. Upload take_0.mp4 (or any clip) as a Kaggle '
    'Dataset and attach via Add data.'
)
INPUT = _candidates[0]
print(f'[input]  {INPUT}')

WORK = '/kaggle/working/pp_4k'; os.makedirs(WORK, exist_ok=True)
FDEN    = f'{WORK}/f_den';    os.makedirs(FDEN, exist_ok=True)
FFSR    = f'{WORK}/f_fsr';    os.makedirs(FFSR, exist_ok=True)
FESR    = f'{WORK}/f_esr';    os.makedirs(FESR, exist_ok=True)
FBLEND  = f'{WORK}/f_blend';  os.makedirs(FBLEND, exist_ok=True)
FIFPS   = f'{WORK}/f_ifps';   os.makedirs(FIFPS, exist_ok=True)
OUT_H264   = '/kaggle/working/take_0_4k.mp4'
OUT_MASTER = '/kaggle/working/take_0_master.mov'

def run(cmd, **kw): return subprocess.run(cmd, check=True, **kw)

# --- probe ---
probe = json.loads(subprocess.run(
    ['ffprobe','-v','quiet','-print_format','json','-show_streams',INPUT],
    capture_output=True, text=True).stdout)
vs   = next(s for s in probe['streams'] if s['codec_type']=='video')
SW, SH = int(vs['width']), int(vs['height'])
FPS    = vs['r_frame_rate']
print(f'[probe]  {SW}×{SH} @ {FPS}fps')
print(f'[target] Real-ESRGAN ×4 → {SW*4}×{SH*4}  →  pad to 3840×2160 (4K)')

# ==============================================================================
# STEP 1/5 — Temporal denoise (hqdn3d)
# ==============================================================================
print('\n=== STEP 1/5 | Temporal denoise (hqdn3d) ===')
DEN = f'{WORK}/denoised.mp4'
run(['ffmpeg','-y','-i',INPUT,'-vf','hqdn3d=4:3:6:4',
     '-c:v','libx264','-crf','18','-preset','fast', DEN])
run(['ffmpeg','-y','-i', DEN, f'{FDEN}/f%05d.png'])
N = len(glob.glob(f'{FDEN}/f*.png'))
print(f'  {N} denoised frames')

# ==============================================================================
# STEP 2/5 — Install GFPGAN + Real-ESRGAN (with basicsr torchvision fix)
# basicsr ships a stale import (functional_tensor removed in torchvision>=0.16)
# We install, patch the one broken line, then flush the module cache.
# ==============================================================================
print('\n=== STEP 2/5 | Install & patch dependencies ===')
try:
    import gfpgan, realesrgan
except ImportError:
    print('  installing gfpgan + realesrgan...')
    run([sys.executable,'-m','pip','install','-q',
         'gfpgan','basicsr','facexlib','realesrgan'])

# Patch basicsr/data/degradations.py for torchvision >= 0.16
for sp in site.getsitepackages() + [site.getusersitepackages()]:
    _bad = os.path.join(sp, 'basicsr', 'data', 'degradations.py')
    if os.path.exists(_bad):
        _src = open(_bad).read()
        if 'functional_tensor' in _src:
            _src = _src.replace(
                'from torchvision.transforms.functional_tensor import rgb_to_grayscale',
                'from torchvision.transforms.functional import rgb_to_grayscale'
            )
            open(_bad, 'w').write(_src)
            print(f'  [patch] basicsr degradations.py fixed')
        break

# Flush stale module cache
for _m in [k for k in sys.modules if k.startswith(('basicsr','gfpgan','realesrgan'))]:
    sys.modules.pop(_m, None)

from gfpgan import GFPGANer
from realesrgan import RealESRGANer
from basicsr.archs.rrdbnet_arch import RRDBNet
import cv2
print('  imports OK')

# ==============================================================================
# STEP 3/5 — GFPGAN face super-resolution (at source resolution, weight=0.7)
# ==============================================================================
print('\n=== STEP 3/5 | GFPGAN face SR (source res) ===')
face_enhancer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.0/GFPGANv1.3.pth',
    upscale=1,
    arch='clean',
    channel_multiplier=2,
    bg_upsampler=None,
)
frames = sorted(glob.glob(f'{FDEN}/f*.png'))
for i, fp in enumerate(frames):
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    _, _, restored = face_enhancer.enhance(
        img, has_aligned=False, only_center_face=False,
        paste_back=True, weight=0.7
    )
    cv2.imwrite(f'{FFSR}/f{i+1:05d}.png', restored if restored is not None else img)
    if (i+1) % 10 == 0 or (i+1) == len(frames):
        print(f'  GFPGAN: {i+1}/{len(frames)}')
del face_enhancer  # free VRAM before Real-ESRGAN
print('  GFPGAN done')

# ==============================================================================
# STEP 4/5 — Real-ESRGAN ×4  (768×512 → 3072×2048)
# tile=512 patches RAM/VRAM; half=True for GPU fp16 speed
# ==============================================================================
print('\n=== STEP 4/5 | Real-ESRGAN ×4 (3072×2048) ===')
import torch
USE_GPU = torch.cuda.is_available()
print(f'  device: {"GPU (fp16)" if USE_GPU else "CPU (fp32) — ~1-2 min/frame, be patient"}')

rrdb = RRDBNet(num_in_ch=3, num_out_ch=3, num_feat=64,
               num_block=23, num_grow_ch=32, scale=4)
upsampler = RealESRGANer(
    scale=4,
    model_path='https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth',
    model=rrdb,
    tile=512,          # process in 512×512 tiles (controls peak RAM)
    tile_pad=10,
    pre_pad=0,
    half=USE_GPU,      # fp16 on GPU, fp32 on CPU
)

fsr_frames = sorted(glob.glob(f'{FFSR}/f*.png'))
for i, fp in enumerate(fsr_frames):
    img = cv2.imread(fp, cv2.IMREAD_COLOR)
    out, _ = upsampler.enhance(img, outscale=4)
    cv2.imwrite(f'{FESR}/f{i+1:05d}.png', out)
    if (i+1) % 5 == 0 or (i+1) == len(fsr_frames):
        print(f'  ESRGAN: {i+1}/{len(fsr_frames)}  [{SW*4}x{SH*4}]')
print('  Real-ESRGAN done')

# ==============================================================================
# STEP 4b -- Temporal-consistency blend (kills Real-ESRGAN per-frame shimmer)
# Each frame = prev*blend + cur*(1-blend). Sweet-spot 0.10-0.20.
# ==============================================================================
print('\n=== STEP 4b | Temporal blend (w={TEMPORAL_BLEND}) ===')
src_after_esr = FESR
if TEMPORAL_BLEND and TEMPORAL_BLEND > 0:
    print(f'  blending (w={TEMPORAL_BLEND})')
    prev = None
    for idx, fp in enumerate(sorted(glob.glob(f'{FESR}/f*.png'))):
        cur = cv2.imread(fp, cv2.IMREAD_COLOR)
        if prev is None:
            out = cur
        else:
            out = cv2.addWeighted(cur, 1.0 - TEMPORAL_BLEND, prev,
                                  TEMPORAL_BLEND, 0)
        cv2.imwrite(f'{FBLEND}/f{idx+1:05d}.png', out)
        prev = cur
    src_after_esr = FBLEND
    print('  temporal blend done')
else:
    print('  skipped (TEMPORAL_BLEND=0)')

# ==============================================================================
# STEP 4c -- RIFE optical-flow interpolation to 24fps (kills choppy 16fps look)
# rife-ncnn-vulkan CLI: doubles frames to target fps using flow-based inference.
# Falls back to ffmpeg minterpolate (mi_mode=blend, non-tearing) if RIFE missing.
# ==============================================================================
final_src_dir = src_after_esr
src_fps = float(FPS.split('/')[0])/float(FPS.split('/')[1]) if '/' in FPS else float(FPS or 16)
if RIFE_TFPS and RIFE_TFPS > 0 and src_fps < RIFE_TFPS - 0.5:
    print(f'\n=== STEP 4c | RIFE -> {RIFE_TFPS}fps (optical-flow) ===')
    rife_bin = shutil.which('rife-ncnn-vulkan')
    if not rife_bin:
        try:
            run([sys.executable,'-m','pip','install','-q','rife-ncnn-vulkan'],
                capture_output=True)
            import rife_ncnn_vulkan as _r
            rife_bin = getattr(_r, 'bin_path', None) or shutil.which('rife-ncnn-vulkan') or ''
        except Exception as _e:
            print(f'  RIFE unavailable: {_e!r} -> ffmpeg blend fallback')
    if rife_bin:
        run([rife_bin, '-i', src_after_esr, '-o', FIFPS,
             '-t', f'r{RIFE_TFPS}', '-n', 'rife-v4.6'], capture_output=True)
        final_src_dir = FIFPS
        src_fps = float(RIFE_TFPS)
    if final_src_dir == src_after_esr:  # RIFE didn't produce output
        print('  using ffmpeg minterpolate (mi_mode=blend, non-tearing) fallback')
        interp_in = f'{WORK}/ifps_in.mp4'
        run(['ffmpeg','-y','-framerate',FPS,'-i',
             f'{src_after_esr}/f%05d.png',
             '-c:v','libx264','-crf','0','-pix_fmt','yuv420p', interp_in],
            capture_output=True)
        run(['ffmpeg','-y','-i',interp_in,
             '-vf',f'minterpolate=fps={RIFE_TFPS}:mi_mode=blend',
             '-c:v','libx264','-crf','0','-pix_fmt','yuv420p',
             f'{FIFPS}/src.mp4'], capture_output=True)
        run(['ffmpeg','-y','-i',f'{FIFPS}/src.mp4',
             f'{FIFPS}/f%05d.png'], capture_output=True)
        final_src_dir = FIFPS
        src_fps = float(RIFE_TFPS)
else:
    print(f'  [skip] source fps already >= {RIFE_TFPS}')

# ==============================================================================
# STEP 5/5 -- Cinematic grade + 4K pad + dual encode (H.264 review + ProRes 4444 XQ)
# ==============================================================================
# --- Per-scene .cube LUT pass (kills uniform-look tell) ---
if SCENE_LUTS:
    print(f'\n=== STEP 4.5 | Per-scene .cube LUT ({len(SCENE_LUTS)} scenes) ===')
    FLUT = f'{WORK}/f_lut'; os.makedirs(FLUT, exist_ok=True)
    all_frames = sorted(glob.glob(f'{final_src_dir}/f*.png'))
    total = len(all_frames)
    # validate the LUT files exist before touching any frame
    for rng, cube in SCENE_LUTS.items():
        if not os.path.exists(cube):
            raise FileNotFoundError(f'SCENE_LUTS references missing .cube: {cube}')
        lo, hi = rng
        if lo < 0 or hi <= lo or hi > total:
            raise ValueError(f'SCENE_LUTS range {rng} invalid for {total} frames')
    # build per-scene concat demux file; each scene is its own ffmpeg pass so
    # the 3dlut filter only applies inside that scene (no global smear).
    scene_clips = []
    for rng, cube in sorted(SCENE_LUTS.items()):
        lo, hi = rng
        seg_in = f'{WORK}/lut_seg_in_{lo}_{hi}.txt'
        with open(seg_in, 'w') as fh:
            for idx in range(lo, hi):
                fh.write(f"file '{all_frames[idx]}'\n")
        seg_out = f'{WORK}/lut_seg_{lo}_{hi}.mov'
        # lut3d accepts .cube directly; format=auto picks the right shoehorn.
        run(['ffmpeg','-y','-safe','0','-f','concat','-i',seg_in,
             '-vf', f"lut3d=file='{cube}':format=auto",
             '-c:v','ffv1','-pix_fmt','bgr0', seg_out], capture_output=True)
        scene_clips.append(seg_out)
        print(f'  scene [{lo}:{hi}] -> {cube}')
    # concat scene clips back to a single PNG sequence at final_src_dir_lut
    concat_list = f'{WORK}/lut_concat.txt'
    with open(concat_list, 'w') as fh:
        for c in scene_clips:
            fh.write(f"file '{c}'\n")
    FLUT_FINAL = f'{WORK}/f_lut_final'; os.makedirs(FLUT_FINAL, exist_ok=True)
    run(['ffmpeg','-y','-safe','0','-f','concat','-i',concat_list,
         f'{FLUT_FINAL}/f%05d.png'], capture_output=True)
    final_src_dir = FLUT_FINAL
    print('  per-scene LUT pass done')

print('\n=== STEP 5/5 | Grade + 4K pad + dual encode ===')
GRADE = (
    "curves="
    "r='0/0 0.45/0.41 1/0.91':"
    "g='0/0 0.5/0.49 1/0.94':"
    "b='0/0.02 0.5/0.53 1/0.97',"
    "eq=contrast=1.06:brightness=-0.02:saturation=0.88"
)
VF = (
    f"{GRADE},"
    f"scale=3840:2160:force_original_aspect_ratio=decrease:flags=lanczos,"
    f"pad=3840:2160:(ow-iw)/2:(oh-ih)/2:black"
)
# A/V sync: if RIFE changed duration, the original audio no longer fits.
orig_fps = float(FPS.split('/')[0])/float(FPS.split('/')[1]) if '/' in FPS else float(FPS or 16)
atempo_chain = ''
if abs(src_fps - orig_fps) > 0.01:
    ratio = orig_fps / src_fps  # >1 = video got longer (RIFE up), slowdown audio
    # ffmpeg atempo range is [0.5, 100.0]; chain for outside-range ratios
    atempos, r = [], ratio
    while r > 100.0:
        atempos.append(100.0); r /= 100.0
    while r < 0.5:
        atempos.append(0.5);  r /= 0.5
    atempos.append(r)
    atempo_chain = ','.join(f'atempo={t:.6f}' for t in atempos)
    print(f'  [audio] atempo={atempo_chain} (orig {orig_fps:.3f} -> video {src_fps:.3f})')
audio_filter = ['-af', atempo_chain] if atempo_chain else []
fps_arg = f'{int(src_fps) if src_fps.is_integer() else src_fps:.3f}'

# H.264 review cut (small, scrubbable)
run(['ffmpeg','-y',
     '-framerate', fps_arg, '-i', f'{final_src_dir}/f%05d.png',
     '-i', INPUT, '-map','0:v','-map','1:a',
     '-vf', VF,
     '-c:v','libx264','-crf','14','-preset','slow',
     *audio_filter,
     '-c:a','aac','-b:a','192k',
     '-pix_fmt','yuv420p', OUT_H264])

# ProRes 4444 XQ master (12-bit, broadcast-grade; what a colorist grades against)
if PRORES_MASTER:
    run(['ffmpeg','-y',
         '-framerate', fps_arg, '-i', f'{final_src_dir}/f%05d.png',
         '-i', INPUT, '-map','0:v','-map','1:a',
         '-vf', VF,
         '-c:v','prores_ks','-profile:v','5','-vendor','apl0',
         '-pix_fmt','yuv444p10le',
         '-colorspace','bt709','-color_primaries','bt709','-color_trc','bt709',
         *audio_filter,
         '-c:a','pcm_s24le',
         OUT_MASTER])

h264_sz = os.path.getsize(OUT_H264)/1e6
print(f'\n[ok]  take_0_4k.mp4 + take_0_master.mov')
print(f'   source    : {SW}x{SH} @ {FPS}fps ({src_fps:.1f}fps)')
print(f'   ESRGAN    : {SW*4}x{SH*4}  (Real-ESRGAN x4)')
print(f'   frame rate: {src_fps:.1f}fps (RIFE {RIFE_TFPS if RIFE_TFPS else "off"})')
print(f'   H.264     : 3840x2160 UHD 4K ({h264_sz:.1f} MB)')
if PRORES_MASTER:
    mov_sz = os.path.getsize(OUT_MASTER)/1e6
    print(f'   ProRes    : 12-bit 4:4:4 XQ master ({mov_sz:.1f} MB)')
print(f'   pipeline  : hqdn3d -> GFPGAN (w=0.7) -> ESRGAN x4 -> '
      f'temporal blend (w={TEMPORAL_BLEND}) -> RIFE {RIFE_TFPS}fps -> '
      f'grade -> 4K pad -> H.264 + ProRes 4444 XQ')

shutil.rmtree(WORK)
print('   scratch cleaned')
